# All Limits Arena starting-squad optimization

This notebook selects the highest-score **11-player starting squad** for a requested Bundesliga matchday. It reads local project data only and makes no API calls.

## Inputs and selection rules

- Score candidates are read from `C:\kickbase project\outputs\expected_points\expected_points_*.csv`. The filename is parsed as `expected_points_{retrieval_timestamp}_{method}_{metric_creation_timestamp}.csv`; timestamps use `YYYYMMDD_HHMMSS_+ZZZZ`. The valid file with the latest **metric-creation timestamp in its filename** is selected. Filesystem timestamps are never used, and methods may contain underscores.
- The matchday is supplied interactively with `int(input(...))` and must be a positive integer.
- Match participants are loaded first from `outputs/sofascore/match_ids/match_ids_{matchday}.json`. If that file is missing or unusable, the notebook falls back to `outputs/fotmob/match_ids/match_ids_{matchday}_fotmob.json`. No website navigation, scraping, or API request occurs.
- The current score schema uses Kickbase team IDs while the match providers use different ID namespaces. A validated 18-club Kickbase mapping and exact provider-name aliases are embedded below. Unknown or ambiguous teams stop execution.

## Optimization model

The model is a cardinality-constrained binary knapsack built with PuLP and solved by CBC. It creates selection and captain binary variables per player. Exactly one selected player is captain, so that player's score is counted once in the squad total and once again as the captain bonus. The primary objective maximizes this captain-doubled total. Among squads with exactly the same optimal captain-doubled total, a second solve minimizes total in-game value without sacrificing score.

Every squad must contain exactly 11 players, exactly 1 GK, the exact DEF/MID/FOR counts of its formation, cost no more than €150,000,000, use no more than 1 player from one club, and use no more than 2 players combined from the two clubs in any match. The ten evaluated formations, in deterministic tie order, are `4-4-2`, `4-2-4`, `3-4-3`, `4-3-3`, `5-3-2`, `3-5-2`, `5-4-1`, `4-5-1`, `3-6-1`, and `5-2-3`. Each formation includes one additional goalkeeper. The globally best feasible formation is chosen by score, then lower cost, then this list order.

Market values are converted to an internal integer-euro representation after explicitly testing whether the source column is expressed in euros, thousands of euros, or millions of euros. Original CSV columns and values remain unchanged. Expected-point coefficients are also integerized exactly for reliable two-stage optimization.

After solving, every constraint and the captain assignment are recalculated independently from the selected dataframe. Only a verified squad is sorted `GK → DEF → MID → FOR` (score descending within each group) and saved to `C:\kickbase project\outputs\optimized_squad\optimized_squad_{method}_{retrieval_timestamp}_{metric_creation_timestamp}_{squad_optimisation_timestamp}.csv`. The notebook output identifies the recommended captain; the export remains exactly the 11 selected rows with all—and only—the original score columns.

Required packages are `pandas`, `IPython`/Jupyter, and `PuLP` with an available CBC solver. Standard-library modules used are `pathlib`, `json`, `datetime`, `decimal`, `re`, `math`, `unicodedata`, `warnings`, and `dataclasses`. The notebook does not install packages automatically.

## 1. Imports, configuration, and domain definitions

This section defines all paths, formations, schema aliases, position rules, team mappings, and lightweight result containers used by the remaining sections.

In [1]:
# Import the libraries required by this notebook step.
from __future__ import annotations

import json
import math
import re
import unicodedata
import warnings
from dataclasses import dataclass
from datetime import datetime
from decimal import Decimal, InvalidOperation
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display

# Handle expected failures with a clear, actionable message.
try:
    from pulp import (
        LpMaximize,
        LpMinimize,
        LpProblem,
        LpStatus,
        LpVariable,
        PULP_CBC_CMD,
        PulpSolverError,
        lpSum,
        value,
    )
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'PuLP is required. Install it in this Jupyter kernel (for example, '
        '`python -m pip install pulp`) and restart the kernel.'
    ) from exc

# Set workflow configuration value: PROJECT_ROOT.
PROJECT_ROOT = Path(r'C:\kickbase project')
# Set workflow configuration value: EXPECTED_POINTS_DIR.
EXPECTED_POINTS_DIR = PROJECT_ROOT / 'outputs' / 'expected_points'
# Set workflow configuration value: SOFASCORE_MATCH_DIR.
SOFASCORE_MATCH_DIR = PROJECT_ROOT / 'outputs' / 'sofascore' / 'match_ids'
# Set workflow configuration value: FOTMOB_MATCH_DIR.
FOTMOB_MATCH_DIR = PROJECT_ROOT / 'outputs' / 'fotmob' / 'match_ids'
# Set workflow configuration value: OPTIMIZED_SQUAD_DIR.
OPTIMIZED_SQUAD_DIR = PROJECT_ROOT / 'outputs' / 'optimized_squad'
# Set workflow configuration value: BUDGET_EUR.
BUDGET_EUR = 150_000_000
# Set workflow configuration value: SQUAD_SIZE.
SQUAD_SIZE = 11
# Set workflow configuration value: MAX_PLAYERS_PER_CLUB.
MAX_PLAYERS_PER_CLUB = 1
# Set workflow configuration value: MAX_PLAYERS_PER_MATCH.
MAX_PLAYERS_PER_MATCH = 2
# Set workflow configuration value: MIN_PLAUSIBLE_PLAYER_VALUE_EUR.
MIN_PLAUSIBLE_PLAYER_VALUE_EUR = 100_000
# Set workflow configuration value: MAX_PLAUSIBLE_PLAYER_VALUE_EUR.
MAX_PLAUSIBLE_PLAYER_VALUE_EUR = 500_000_000

# Set workflow configuration value: ALLOWED_FORMATIONS.
ALLOWED_FORMATIONS = {
    '4-4-2': {'DEF': 4, 'MID': 4, 'FOR': 2},
    '4-2-4': {'DEF': 4, 'MID': 2, 'FOR': 4},
    '3-4-3': {'DEF': 3, 'MID': 4, 'FOR': 3},
    '4-3-3': {'DEF': 4, 'MID': 3, 'FOR': 3},
    '5-3-2': {'DEF': 5, 'MID': 3, 'FOR': 2},
    '3-5-2': {'DEF': 3, 'MID': 5, 'FOR': 2},
    '5-4-1': {'DEF': 5, 'MID': 4, 'FOR': 1},
    '4-5-1': {'DEF': 4, 'MID': 5, 'FOR': 1},
    '3-6-1': {'DEF': 3, 'MID': 6, 'FOR': 1},
    '5-2-3': {'DEF': 5, 'MID': 2, 'FOR': 3},
}

# Set workflow configuration value: COLUMN_ALIASES.
COLUMN_ALIASES = {
    'player_id': {'id', 'player_id', 'playerId'},
    'player_name': {'name', 'player_name', 'full_name', 'fullName'},
    'score': {'score'},
    'market_value': {'marketValue', 'market_value', 'ingame_value', 'in_game_value'},
    'club': {'teamId', 'team_id', 'club_id', 'club', 'team'},
    'position': {'position', 'player_position', 'ingame_position', 'kbstats_position'},
}

# Set workflow configuration value: POSITION_ALIASES.
POSITION_ALIASES = {
    '1': 'GK', 'gk': 'GK', 'goalkeeper': 'GK', 'keeper': 'GK',
    '2': 'DEF', 'def': 'DEF', 'defender': 'DEF', 'defence': 'DEF', 'defense': 'DEF',
    '3': 'MID', 'mid': 'MID', 'midfielder': 'MID', 'midfield': 'MID',
    '4': 'FOR', 'for': 'FOR', 'fwd': 'FOR', 'fw': 'FOR', 'forward': 'FOR',
    'striker': 'FOR', 'attacker': 'FOR',
}

# Set workflow configuration value: KB_TEAM_ID_TO_KEY.
KB_TEAM_ID_TO_KEY = {
    2: 'bayern', 3: 'dortmund', 4: 'frankfurt', 5: 'freiburg',
    6: 'hamburg', 7: 'leverkusen', 8: 'schalke', 9: 'stuttgart',
    10: 'bremen', 13: 'augsburg', 14: 'hoffenheim', 15: 'gladbach',
    18: 'mainz', 28: 'koeln', 29: 'paderborn', 40: 'union',
    43: 'leipzig', 77: 'elversberg',
}

# Set workflow configuration value: TEAM_DISPLAY_NAMES.
TEAM_DISPLAY_NAMES = {
    'bayern': 'FC Bayern München',
    'stuttgart': 'VfB Stuttgart',
    'koeln': '1. FC Köln',
    'hoffenheim': 'TSG Hoffenheim',
    'union': '1. FC Union Berlin',
    'frankfurt': 'Eintracht Frankfurt',
    'mainz': '1. FSV Mainz 05',
    'paderborn': 'SC Paderborn 07',
    'dortmund': 'Borussia Dortmund',
    'hamburg': 'Hamburger SV',
    'leipzig': 'RB Leipzig',
    'gladbach': 'Borussia Mönchengladbach',
    'freiburg': 'SC Freiburg',
    'bremen': 'SV Werder Bremen',
    'elversberg': 'SV 07 Elversberg',
    'leverkusen': 'Bayer 04 Leverkusen',
    'augsburg': 'FC Augsburg',
    'schalke': 'FC Schalke 04',
}

# Set workflow configuration value: TEAM_ALIASES.
TEAM_ALIASES = {
    'bayern': {'FC Bayern München', 'Bayern München', 'Bayern Munich'},
    'stuttgart': {'VfB Stuttgart'},
    'koeln': {'1. FC Köln', 'FC Köln', '1. FC Cologne', 'FC Cologne'},
    'hoffenheim': {'TSG Hoffenheim', 'Hoffenheim'},
    'union': {'1. FC Union Berlin', 'Union Berlin'},
    'frankfurt': {'Eintracht Frankfurt'},
    'mainz': {'1. FSV Mainz 05', 'Mainz 05'},
    'paderborn': {'SC Paderborn 07', 'SC Paderborn', 'Paderborn'},
    'dortmund': {'Borussia Dortmund'},
    'hamburg': {'Hamburger SV', 'Hamburg'},
    'leipzig': {'RB Leipzig'},
    'gladbach': {"Borussia M'gladbach", 'Borussia Mönchengladbach', 'Mönchengladbach'},
    'freiburg': {'SC Freiburg', 'Freiburg'},
    'bremen': {'SV Werder Bremen', 'Werder Bremen'},
    'elversberg': {'SV 07 Elversberg', 'SV Elversberg', 'Elversberg'},
    'leverkusen': {'Bayer 04 Leverkusen', 'Bayer Leverkusen'},
    'augsburg': {'FC Augsburg', 'Augsburg'},
    'schalke': {'FC Schalke 04', 'Schalke 04'},
}

# Set workflow configuration value: TIMESTAMP_PATTERN.
TIMESTAMP_PATTERN = r'\d{8}_\d{6}_[+-]\d{4}'
# Set workflow configuration value: EXPECTED_POINTS_FILENAME_RE.
EXPECTED_POINTS_FILENAME_RE = re.compile(
    rf'^expected_points_(?P<retrieval>{TIMESTAMP_PATTERN})_'
    rf'(?P<method>.+)_(?P<metric>{TIMESTAMP_PATTERN})\.csv$'
)
# Set workflow configuration value: TIMESTAMP_FORMAT.
TIMESTAMP_FORMAT = '%Y%m%d_%H%M%S_%z'

# Process each available item while preserving the current workflow state.
for formation_name, counts in ALLOWED_FORMATIONS.items():
    # Validate the input before continuing with later processing.
    if sum(counts.values()) != 10:
        raise ValueError(f'Formation {formation_name} does not contain 10 outfield players.')

@dataclass(frozen=True)
# Define Score Metadata to keep related behaviour explicit.
class ScoreMetadata:
    path: Path
    retrieval_timestamp: str
    method: str
    metric_creation_timestamp: str
    retrieval_datetime: datetime
    metric_creation_datetime: datetime

@dataclass(frozen=True)
# Define Match Record to keep related behaviour explicit.
class MatchRecord:
    match_id: int
    home_team_id: int
    home_team_name: str
    away_team_id: int
    away_team_name: str

@dataclass(frozen=True)
# Define Mapped Match to keep related behaviour explicit.
class MappedMatch:
    record: MatchRecord
    home_key: str
    away_key: str

@dataclass
# Define Prepared Data to keep related behaviour explicit.
class PreparedData:
    df: pd.DataFrame
    original_columns: list[str]
    columns: dict[str, str]
    positions: pd.Series
    score_numeric: pd.Series
    score_units: pd.Series
    score_scale: int
    value_numeric: pd.Series
    value_eur: pd.Series
    value_unit: str
    team_keys: pd.Series
    team_raw_to_key: dict[str, str]

@dataclass(frozen=True)
# Define Formation Result to keep related behaviour explicit.
class FormationResult:
    formation: str
    status: str
    chosen_indices: tuple[int, ...] = ()
    captain_index: int | None = None
    total_score_units: int | None = None
    total_value_eur: int | None = None

## 2. Score discovery and schema preparation

The functions below parse timestamps from filenames, select the newest valid metric, identify required source columns without renaming them, validate every player row, normalize positions, and create exact numerical helper series.

In [2]:
# Parse and validate score filename for reuse in the workflow.
def parse_score_filename(path: Path) -> ScoreMetadata:
    match = EXPECTED_POINTS_FILENAME_RE.fullmatch(path.name)
    # Validate the input before continuing with later processing.
    if match is None:
        raise ValueError('filename does not match the required timestamp structure')

    retrieval_timestamp = match.group('retrieval')
    method = match.group('method')
    metric_timestamp = match.group('metric')
    # Validate the input before continuing with later processing.
    if not method.strip():
        raise ValueError('method is empty')

    # Handle expected failures with a clear, actionable message.
    try:
        retrieval_datetime = datetime.strptime(retrieval_timestamp, TIMESTAMP_FORMAT)
        metric_datetime = datetime.strptime(metric_timestamp, TIMESTAMP_FORMAT)
    except ValueError as exc:
        raise ValueError(f'unparseable filename timestamp: {exc}') from exc

    return ScoreMetadata(
        path=path,
        retrieval_timestamp=retrieval_timestamp,
        method=method,
        metric_creation_timestamp=metric_timestamp,
        retrieval_datetime=retrieval_datetime,
        metric_creation_datetime=metric_datetime,
    )


# Find the latest score input for reuse in the workflow.
def discover_latest_score(directory: Path) -> ScoreMetadata:
    # Validate the input before continuing with later processing.
    if not directory.is_dir():
        raise FileNotFoundError(f'Score-input directory does not exist: {directory}')

    candidates = sorted(directory.glob('expected_points_*.csv'))
    # Validate the input before continuing with later processing.
    if not candidates:
        raise FileNotFoundError(f'No score-input files (expected_points_*.csv) found in: {directory}')

    valid: list[ScoreMetadata] = []
    # Process each available item while preserving the current workflow state.
    for path in candidates:
        # Handle expected failures with a clear, actionable message.
        try:
            valid.append(parse_score_filename(path))
        except ValueError as exc:
            warnings.warn(f'Ignoring malformed score-input file {path.name!r}: {exc}')

    # Validate the input before continuing with later processing.
    if not valid:
        raise FileNotFoundError(
            f'No valid score-input CSV remains in {directory}; check filename timestamps.'
        )

    latest_datetime = max(item.metric_creation_datetime for item in valid)
    newest = [item for item in valid if item.metric_creation_datetime == latest_datetime]
    # Validate the input before continuing with later processing.
    if len(newest) != 1:
        names = ', '.join(item.path.name for item in newest)
        raise ValueError(
            'Score-input selection is ambiguous: multiple files have the latest '
            f'metric-creation timestamp {latest_datetime.isoformat()}: {names}'
        )
    return newest[0]


# Normalize column name for reuse in the workflow.
def normalize_column_name(value: str) -> str:
    return re.sub(r'[^a-z0-9]+', '', str(value).casefold())


# Handle required columns for reuse in the workflow.
def identify_required_columns(columns: list[str]) -> dict[str, str]:
    normalized_actual: dict[str, list[str]] = {}
    # Process each available item while preserving the current workflow state.
    for column in columns:
        normalized_actual.setdefault(normalize_column_name(column), []).append(column)

    resolved: dict[str, str] = {}
    # Process each available item while preserving the current workflow state.
    for logical_name, aliases in COLUMN_ALIASES.items():
        normalized_aliases = {normalize_column_name(alias) for alias in aliases}
        matches = [
            column
            for normalized, actual_columns in normalized_actual.items()
            if normalized in normalized_aliases
            for column in actual_columns
        ]
        # Validate the input before continuing with later processing.
        if len(matches) != 1:
            available = ', '.join(repr(column) for column in columns)
            raise ValueError(
                f'Could not identify exactly one {logical_name!r} column. '
                f'Matches={matches}; available columns=[{available}]'
            )
        resolved[logical_name] = matches[0]
    return resolved


# Handle row numbers for reuse in the workflow.
def source_row_numbers(indices: list[int], limit: int = 10) -> str:
    rows = [str(index + 2) for index in indices[:limit]]
    suffix = ' ...' if len(indices) > limit else ''
    return ', '.join(rows) + suffix


# Parse and validate decimal series for reuse in the workflow.
def parse_decimal_series(series: pd.Series, label: str) -> tuple[list[Decimal], pd.Series]:
    decimals: list[Decimal] = []
    missing: list[int] = []
    invalid: list[int] = []

    # Process each available item while preserving the current workflow state.
    for index, raw_value in series.items():
        text = str(raw_value).strip()
        if not text:
            missing.append(int(index))
            decimals.append(Decimal('NaN'))
            continue
        # Handle expected failures with a clear, actionable message.
        try:
            parsed = Decimal(text)
        except InvalidOperation:
            invalid.append(int(index))
            decimals.append(Decimal('NaN'))
            continue
        if not parsed.is_finite():
            invalid.append(int(index))
        decimals.append(parsed)

    # Validate the input before continuing with later processing.
    if missing:
        raise ValueError(
            f'{label} contains missing values at source CSV row(s): '
            f'{source_row_numbers(missing)}'
        )
    # Validate the input before continuing with later processing.
    if invalid:
        samples = [repr(series.loc[index]) for index in invalid[:5]]
        raise ValueError(
            f'{label} contains non-numeric or non-finite values at source CSV row(s) '
            f'{source_row_numbers(invalid)}; sample values={samples}'
        )

    numeric = pd.Series([float(item) for item in decimals], index=series.index, dtype='float64')
    return decimals, numeric


# Integerize score values for reuse in the workflow.
def integerize_scores(decimals: list[Decimal], index: pd.Index) -> tuple[pd.Series, int]:
    normalized = [item.normalize() if item != 0 else Decimal(0) for item in decimals]
    decimal_places = max(max(0, -item.as_tuple().exponent) for item in normalized)
    scale = 10 ** decimal_places
    units: list[int] = []
    # Process each available item while preserving the current workflow state.
    for item in decimals:
        scaled = item * scale
        # Validate the input before continuing with later processing.
        if scaled != scaled.to_integral_value():
            raise ValueError(f'Could not integerize score value {item!r} exactly.')
        units.append(int(scaled))
    return pd.Series(units, index=index, dtype=object), scale


# Normalize market values to euros for reuse in the workflow.
def normalize_market_values_to_euros(
    decimals: list[Decimal], index: pd.Index
) -> tuple[pd.Series, str]:
    # Validate the input before continuing with later processing.
    if any(item <= 0 for item in decimals):
        bad = [position for position, item in enumerate(decimals) if item <= 0]
        raise ValueError(
            'Market values must be positive; invalid source CSV row(s): '
            f'{source_row_numbers(bad)}'
        )

    interpretations = (
        ('euros', Decimal(1)),
        ('thousands of euros', Decimal(1_000)),
        ('millions of euros', Decimal(1_000_000)),
    )
    plausible: list[tuple[str, list[int]]] = []
    diagnostics: list[str] = []

    # Process each available item while preserving the current workflow state.
    for unit_name, factor in interpretations:
        scaled = [item * factor for item in decimals]
        if not all(item == item.to_integral_value() for item in scaled):
            diagnostics.append(f'{unit_name}: would produce fractional euros')
            continue
        integer_values = [int(item) for item in scaled]
        minimum = min(integer_values)
        maximum = max(integer_values)
        # Choose the appropriate path for the current data state.
        if (
            minimum >= MIN_PLAUSIBLE_PLAYER_VALUE_EUR
            and maximum <= MAX_PLAUSIBLE_PLAYER_VALUE_EUR
        ):
            plausible.append((unit_name, integer_values))
        else:
            diagnostics.append(
                f'{unit_name}: interpreted range €{minimum:,} to €{maximum:,} is outside '
                f'€{MIN_PLAUSIBLE_PLAYER_VALUE_EUR:,} to '
                f'€{MAX_PLAUSIBLE_PLAYER_VALUE_EUR:,}'
            )

    # Validate the input before continuing with later processing.
    if len(plausible) != 1:
        raw_min = min(decimals)
        raw_max = max(decimals)
        raise ValueError(
            'Market-value unit is ambiguous or inconsistent. Expected exactly one plausible '
            f'interpretation for raw range {raw_min} to {raw_max}; candidates='
            f'{[item[0] for item in plausible]}; diagnostics={diagnostics}'
        )

    unit_name, integer_values = plausible[0]
    return pd.Series(integer_values, index=index, dtype=object), unit_name


# Normalize position for reuse in the workflow.
def normalize_position(raw_value: Any) -> str:
    text = str(raw_value).strip().casefold()
    # Handle expected failures with a clear, actionable message.
    try:
        numeric = Decimal(text)
    except InvalidOperation:
        numeric = None
    if numeric is not None and numeric.is_finite() and numeric == numeric.to_integral_value():
        text = str(int(numeric))
    # Validate the input before continuing with later processing.
    if text not in POSITION_ALIASES:
        raise ValueError(f'unsupported KBStats position value {raw_value!r}')
    return POSITION_ALIASES[text]


# Load and validate player data for reuse in the workflow.
def load_and_validate_player_data(path: Path) -> dict[str, Any]:
    # Handle expected failures with a clear, actionable message.
    try:
        df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    except pd.errors.EmptyDataError as exc:
        raise ValueError(f'Score CSV is empty: {path}') from exc
    except (OSError, UnicodeError, pd.errors.ParserError) as exc:
        raise ValueError(f'Could not read score CSV {path}: {exc}') from exc

    # Validate the input before continuing with later processing.
    if df.empty:
        raise ValueError(f'Score CSV contains no player rows: {path}')
    df = df.reset_index(drop=True)
    original_columns = list(df.columns)
    columns = identify_required_columns(original_columns)

    id_values = df[columns['player_id']].astype(str).str.strip()
    missing_ids = id_values.index[id_values.eq('')].tolist()
    # Validate the input before continuing with later processing.
    if missing_ids:
        raise ValueError(f'Missing player IDs at source CSV row(s): {source_row_numbers(missing_ids)}')
    duplicate_ids = id_values[id_values.duplicated(keep=False)]
    # Validate the input before continuing with later processing.
    if not duplicate_ids.empty:
        raise ValueError(
            f'Player IDs must be unique; duplicates={sorted(duplicate_ids.unique().tolist())}'
        )

    names = df[columns['player_name']].astype(str).str.strip()
    missing_names = names.index[names.eq('')].tolist()
    # Validate the input before continuing with later processing.
    if missing_names:
        raise ValueError(
            f'Missing player full names at source CSV row(s): {source_row_numbers(missing_names)}'
        )

    raw_clubs = df[columns['club']].astype(str).str.strip()
    missing_clubs = raw_clubs.index[raw_clubs.eq('')].tolist()
    # Validate the input before continuing with later processing.
    if missing_clubs:
        raise ValueError(
            f'Missing club/team values at source CSV row(s): {source_row_numbers(missing_clubs)}'
        )

    positions: list[str] = []
    position_errors: list[str] = []
    # Process each available item while preserving the current workflow state.
    for index, raw_value in df[columns['position']].items():
        # Handle expected failures with a clear, actionable message.
        try:
            positions.append(normalize_position(raw_value))
        except ValueError as exc:
            position_errors.append(f'row {index + 2}: {exc}')
            positions.append('')
    # Validate the input before continuing with later processing.
    if position_errors:
        raise ValueError('Unsupported positions: ' + '; '.join(position_errors[:10]))
    position_series = pd.Series(positions, index=df.index, dtype='string')

    score_decimals, score_numeric = parse_decimal_series(
        df[columns['score']], 'Score'
    )
    score_units, score_scale = integerize_scores(score_decimals, df.index)
    value_decimals, value_numeric = parse_decimal_series(
        df[columns['market_value']], 'Market values'
    )
    value_eur, value_unit = normalize_market_values_to_euros(value_decimals, df.index)

    position_counts = position_series.value_counts().to_dict()
    constructible = [
        name
        for name, counts in ALLOWED_FORMATIONS.items()
        if position_counts.get('GK', 0) >= 1
        and all(position_counts.get(position, 0) >= required for position, required in counts.items())
    ]
    # Validate the input before continuing with later processing.
    if len(df) < 11 or not constructible:
        raise ValueError(
            'Insufficient eligible players to construct any permitted formation. '
            f'Rows={len(df)}; position counts={position_counts}'
        )

    return {
        'df': df,
        'original_columns': original_columns,
        'columns': columns,
        'positions': position_series,
        'score_numeric': score_numeric,
        'score_units': score_units,
        'score_scale': score_scale,
        'value_numeric': value_numeric,
        'value_eur': value_eur,
        'value_unit': value_unit,
    }

## 3. Matchday input and local match-file loading

The matchday is requested interactively. SofaScore is attempted first; any missing, unreadable, malformed, or semantically invalid SofaScore file triggers the documented FotMob fallback. Both current flat JSON records and a small set of unambiguous common wrappers/nested team objects are supported.

In [3]:
# Handle matchday for reuse in the workflow.
def request_matchday() -> int:
    # Handle expected failures with a clear, actionable message.
    try:
        matchday = int(input('Enter the matchday to optimise the squad for: '))
    except ValueError as exc:
        raise ValueError('Matchday must be entered as a positive integer.') from exc
    # Validate the input before continuing with later processing.
    if matchday < 1:
        raise ValueError(f'Matchday must be a positive integer; received {matchday}.')
    return matchday


# Handle positive integer for reuse in the workflow.
def require_positive_integer(value_to_check: Any, label: str) -> int:
    # Validate the input before continuing with later processing.
    if isinstance(value_to_check, bool):
        raise ValueError(f'{label} must be a positive integer, not boolean.')
    # Validate the input before continuing with later processing.
    if isinstance(value_to_check, int):
        parsed = value_to_check
    # Validate the input before continuing with later processing.
    elif isinstance(value_to_check, str) and value_to_check.strip().isdigit():
        parsed = int(value_to_check.strip())
    # Validate the input before continuing with later processing.
    elif isinstance(value_to_check, float) and value_to_check.is_integer():
        parsed = int(value_to_check)
    else:
        raise ValueError(f'{label} must be a positive integer; received {value_to_check!r}.')
    # Validate the input before continuing with later processing.
    if parsed < 1:
        raise ValueError(f'{label} must be greater than zero; received {parsed}.')
    return parsed


# Handle field for reuse in the workflow.
def unique_field(record: dict[str, Any], aliases: tuple[str, ...], label: str) -> Any:
    present = [(key, record[key]) for key in aliases if key in record and record[key] is not None]
    # Validate the input before continuing with later processing.
    if not present:
        raise ValueError(f'Missing {label}; accepted fields={aliases}.')
    first_value = present[0][1]
    # Validate the input before continuing with later processing.
    if any(candidate != first_value for _, candidate in present[1:]):
        raise ValueError(f'Conflicting {label} fields: {present}.')
    return first_value


# Extract match list for reuse in the workflow.
def extract_match_list(payload: Any) -> list[Any]:
    if isinstance(payload, list):
        return payload
    # Validate the input before continuing with later processing.
    if not isinstance(payload, dict):
        raise ValueError('Match JSON must contain a top-level list or object wrapper.')
    list_fields = [(key, payload[key]) for key in ('matches', 'fixtures', 'events') if isinstance(payload.get(key), list)]
    # Validate the input before continuing with later processing.
    if len(list_fields) != 1:
        raise ValueError(
            'Match JSON object must contain exactly one list field named matches, fixtures, '
            f'or events; found {[key for key, _ in list_fields]}.'
        )
    return list_fields[0][1]


# Extract team side for reuse in the workflow.
def extract_team_side(record: dict[str, Any], side: str, match_number: int) -> tuple[int, str]:
    nested_candidates = [
        record[key]
        for key in (side, f'{side}Team', f'{side}_team')
        if isinstance(record.get(key), dict)
    ]
    # Validate the input before continuing with later processing.
    if len(nested_candidates) > 1 and any(item != nested_candidates[0] for item in nested_candidates[1:]):
        raise ValueError(f'Match {match_number} has conflicting nested {side}-team objects.')

    # Choose the appropriate path for the current data state.
    if nested_candidates:
        team_object = nested_candidates[0]
        name = unique_field(team_object, ('name', 'team', 'team_name', 'teamName'), f'{side} team name')
        team_id = unique_field(team_object, ('id', 'team_id', 'teamId'), f'{side} team ID')
    else:
        name = unique_field(
            record,
            (f'{side}_team', f'{side}Team', f'{side}_team_name', f'{side}TeamName'),
            f'{side} team name',
        )
        team_id = unique_field(
            record,
            (f'{side}_team_id', f'{side}TeamId', f'{side}TeamID'),
            f'{side} team ID',
        )

    # Validate the input before continuing with later processing.
    if not isinstance(name, str) or not name.strip():
        raise ValueError(f'Match {match_number} {side} team name is empty or non-text.')
    return require_positive_integer(team_id, f'Match {match_number} {side} team ID'), name.strip()


# Parse and validate match records for reuse in the workflow.
def parse_match_records(payload: Any) -> list[MatchRecord]:
    raw_matches = extract_match_list(payload)
    # Validate the input before continuing with later processing.
    if not raw_matches:
        raise ValueError('Match JSON contains no matches.')

    matches: list[MatchRecord] = []
    seen_match_ids: set[int] = set()
    provider_team_names: dict[int, str] = {}

    # Process each available item while preserving the current workflow state.
    for match_number, raw_match in enumerate(raw_matches, start=1):
        # Validate the input before continuing with later processing.
        if not isinstance(raw_match, dict):
            raise ValueError(f'Match {match_number} is not a JSON object.')
        match_id = require_positive_integer(
            unique_field(raw_match, ('match_id', 'matchId', 'id'), 'match ID'),
            f'Match {match_number} match ID',
        )
        # Validate the input before continuing with later processing.
        if match_id in seen_match_ids:
            raise ValueError(f'Duplicate match ID in match JSON: {match_id}.')
        seen_match_ids.add(match_id)

        home_id, home_name = extract_team_side(raw_match, 'home', match_number)
        away_id, away_name = extract_team_side(raw_match, 'away', match_number)
        # Validate the input before continuing with later processing.
        if home_id == away_id:
            raise ValueError(f'Match {match_id} uses the same provider team ID for both sides.')

        # Process each available item while preserving the current workflow state.
        for team_id, team_name in ((home_id, home_name), (away_id, away_name)):
            normalized = normalize_team_name(team_name)
            previous = provider_team_names.get(team_id)
            # Validate the input before continuing with later processing.
            if previous is not None and previous != normalized:
                raise ValueError(
                    f'Provider team ID {team_id} has conflicting names in the match JSON.'
                )
            provider_team_names[team_id] = normalized

        matches.append(MatchRecord(match_id, home_id, home_name, away_id, away_name))
    return matches


# Load matchday matches for reuse in the workflow.
def load_matchday_matches(matchday: int) -> tuple[list[MatchRecord], str, Path]:
    attempts = (
        ('SofaScore', SOFASCORE_MATCH_DIR / f'match_ids_{matchday}.json'),
        ('FotMob', FOTMOB_MATCH_DIR / f'match_ids_{matchday}_fotmob.json'),
    )
    errors: list[str] = []

    # Process each available item while preserving the current workflow state.
    for source_name, path in attempts:
        # Handle expected failures with a clear, actionable message.
        try:
            payload = json.loads(path.read_text(encoding='utf-8-sig'))
            matches = parse_match_records(payload)
        except (OSError, UnicodeError, json.JSONDecodeError, TypeError, ValueError) as exc:
            errors.append(f'{source_name}: {path} -> {type(exc).__name__}: {exc}')
            if source_name == 'SofaScore':
                print(f'Warning: SofaScore match file could not be used ({exc}); trying FotMob.')
            continue
        print(f'Match source used: {source_name} ({path})')
        return matches, source_name, path

    attempted_paths = '\n'.join(f'  - {path}' for _, path in attempts)
    error_text = '\n'.join(f'  - {item}' for item in errors)
    raise RuntimeError(
        f'Could not load matchday {matchday} from either local source.\n'
        f'Attempted paths:\n{attempted_paths}\nErrors:\n{error_text}'
    )

## 4. Club normalization, cross-provider mapping, and diagnostics

Provider names are resolved only through exact normalized aliases. Compatible provider IDs are preferred when the CSV ID namespace matches the match file; otherwise the validated Kickbase-ID bridge is used. Every dataset club must map to exactly one match before any model is created.

In [4]:
# Normalize team name for reuse in the workflow.
def normalize_team_name(value_to_normalize: str) -> str:
    normalized = unicodedata.normalize('NFKC', value_to_normalize).casefold().strip()
    normalized = normalized.replace('’', "'").replace('`', "'")
    normalized = re.sub(r'[^\w]+', ' ', normalized, flags=re.UNICODE)
    return ' '.join(normalized.split())


# Build team alias registry for reuse in the workflow.
def build_team_alias_registry() -> dict[str, str]:
    # Validate the input before continuing with later processing.
    if set(KB_TEAM_ID_TO_KEY.values()) != set(TEAM_DISPLAY_NAMES):
        raise ValueError('Embedded Kickbase team map and display-name map are inconsistent.')
    # Validate the input before continuing with later processing.
    if set(TEAM_ALIASES) != set(TEAM_DISPLAY_NAMES):
        raise ValueError('Embedded team aliases and display-name map are inconsistent.')

    registry: dict[str, str] = {}
    # Process each available item while preserving the current workflow state.
    for team_key, aliases in TEAM_ALIASES.items():
        # Process each available item while preserving the current workflow state.
        for alias in set(aliases) | {TEAM_DISPLAY_NAMES[team_key]}:
            normalized = normalize_team_name(alias)
            previous = registry.get(normalized)
            # Validate the input before continuing with later processing.
            if previous is not None and previous != team_key:
                raise ValueError(
                    f'Team alias {alias!r} is ambiguous between {previous!r} and {team_key!r}.'
                )
            registry[normalized] = team_key
    return registry


# Set workflow configuration value: TEAM_ALIAS_TO_KEY.
TEAM_ALIAS_TO_KEY = build_team_alias_registry()


# Resolve team name for reuse in the workflow.
def resolve_team_name(name: str) -> str:
    normalized = normalize_team_name(name)
    # Validate the input before continuing with later processing.
    if normalized not in TEAM_ALIAS_TO_KEY:
        raise ValueError(
            f'Unrecognized team name {name!r} after exact normalization to {normalized!r}.'
        )
    return TEAM_ALIAS_TO_KEY[normalized]


# Handle like for reuse in the workflow.
def integer_like(value_to_parse: Any) -> int | None:
    # Handle expected failures with a clear, actionable message.
    try:
        parsed = Decimal(str(value_to_parse).strip())
    except InvalidOperation:
        return None
    if not parsed.is_finite() or parsed != parsed.to_integral_value() or parsed < 1:
        return None
    return int(parsed)


# Map clubs to matches for reuse in the workflow.
def map_clubs_to_matches(
    df: pd.DataFrame, club_column: str, matches: list[MatchRecord]
) -> tuple[pd.Series, dict[str, str], list[MappedMatch], pd.DataFrame]:
    provider_id_to_key: dict[int, str] = {}
    mapped_matches: list[MappedMatch] = []

    # Process each available item while preserving the current workflow state.
    for record in matches:
        home_key = resolve_team_name(record.home_team_name)
        away_key = resolve_team_name(record.away_team_name)
        # Validate the input before continuing with later processing.
        if home_key == away_key:
            raise ValueError(f'Match {record.match_id} maps both sides to {home_key!r}.')
        # Process each available item while preserving the current workflow state.
        for provider_id, team_key in (
            (record.home_team_id, home_key),
            (record.away_team_id, away_key),
        ):
            previous = provider_id_to_key.get(provider_id)
            # Validate the input before continuing with later processing.
            if previous is not None and previous != team_key:
                raise ValueError(
                    f'Provider team ID {provider_id} maps to both {previous!r} and {team_key!r}.'
                )
            provider_id_to_key[provider_id] = team_key
        mapped_matches.append(MappedMatch(record, home_key, away_key))

    raw_series = df[club_column].astype(str).str.strip()
    unique_raw = list(dict.fromkeys(raw_series.tolist()))
    parsed_ids = {raw: integer_like(raw) for raw in unique_raw}
    all_numeric = all(parsed is not None for parsed in parsed_ids.values())
    raw_to_key: dict[str, str] = {}

    # Validate the input before continuing with later processing.
    if all_numeric and {int(value) for value in parsed_ids.values()} <= set(provider_id_to_key):
        mapping_mode = 'compatible provider team IDs'
        raw_to_key = {raw: provider_id_to_key[int(parsed_ids[raw])] for raw in unique_raw}
    # Validate the input before continuing with later processing.
    elif all_numeric and {int(value) for value in parsed_ids.values()} <= set(KB_TEAM_ID_TO_KEY):
        mapping_mode = 'embedded Kickbase team-ID bridge'
        raw_to_key = {raw: KB_TEAM_ID_TO_KEY[int(parsed_ids[raw])] for raw in unique_raw}
    else:
        mapping_mode = 'exact normalized team names'
        errors: list[str] = []
        # Process each available item while preserving the current workflow state.
        for raw in unique_raw:
            # Handle expected failures with a clear, actionable message.
            try:
                raw_to_key[raw] = resolve_team_name(raw)
            except ValueError as exc:
                errors.append(str(exc))
        # Validate the input before continuing with later processing.
        if errors:
            raise ValueError(
                'Club/team mapping failed. CSV identifiers are neither a compatible provider '
                'ID set, the known Kickbase ID set, nor recognized exact team names: '
                + '; '.join(errors)
            )

    team_keys = raw_series.map(raw_to_key)
    # Validate the input before continuing with later processing.
    if team_keys.isna().any():
        raise ValueError('Internal club mapping error left one or more player rows unmapped.')

    match_counts: dict[str, int] = {}
    # Process each available item while preserving the current workflow state.
    for match in mapped_matches:
        # Process each available item while preserving the current workflow state.
        for team_key in (match.home_key, match.away_key):
            match_counts[team_key] = match_counts.get(team_key, 0) + 1
    bad_counts = {team: count for team, count in match_counts.items() if count != 1}
    # Validate the input before continuing with later processing.
    if bad_counts:
        raise ValueError(f'Clubs mapped to an unexpected number of matches: {bad_counts}')

    dataset_keys = set(team_keys.tolist())
    match_keys = set(match_counts)
    # Validate the input before continuing with later processing.
    if dataset_keys != match_keys:
        missing_from_csv = sorted(match_keys - dataset_keys)
        missing_from_matches = sorted(dataset_keys - match_keys)
        raise ValueError(
            'CSV clubs and matchday clubs do not form a one-to-one matchday mapping. '
            f'Match clubs absent from CSV={missing_from_csv}; '
            f'CSV clubs absent from matches={missing_from_matches}.'
        )

    key_to_raw_values: dict[str, set[str]] = {}
    # Process each available item while preserving the current workflow state.
    for raw, team_key in raw_to_key.items():
        key_to_raw_values.setdefault(team_key, set()).add(raw)
    ambiguous_raw = {key: sorted(values) for key, values in key_to_raw_values.items() if len(values) != 1}
    # Validate the input before continuing with later processing.
    if ambiguous_raw:
        raise ValueError(f'Canonical clubs map to multiple CSV club values: {ambiguous_raw}')
    key_to_raw = {key: next(iter(values)) for key, values in key_to_raw_values.items()}

    diagnostic_rows: list[dict[str, str | int]] = []
    # Process each available item while preserving the current workflow state.
    for match in mapped_matches:
        diagnostic_rows.append(
            {
                'Match ID': match.record.match_id,
                'JSON Home Team': match.record.home_team_name,
                'CSV Home Club': f'{TEAM_DISPLAY_NAMES[match.home_key]} ({key_to_raw[match.home_key]})',
                'JSON Away Team': match.record.away_team_name,
                'CSV Away Club': f'{TEAM_DISPLAY_NAMES[match.away_key]} ({key_to_raw[match.away_key]})',
            }
        )
    mapping_df = pd.DataFrame(diagnostic_rows)
    print(f'Club mapping mode: {mapping_mode}')
    return team_keys.astype('string'), raw_to_key, mapped_matches, mapping_df


# Prepare optimization data for reuse in the workflow.
def prepare_optimization_data(
    path: Path, matches: list[MatchRecord]
) -> tuple[PreparedData, list[MappedMatch], pd.DataFrame]:
    base = load_and_validate_player_data(path)
    team_keys, raw_to_key, mapped_matches, mapping_df = map_clubs_to_matches(
        base['df'], base['columns']['club'], matches
    )
    prepared = PreparedData(
        df=base['df'],
        original_columns=base['original_columns'],
        columns=base['columns'],
        positions=base['positions'],
        score_numeric=base['score_numeric'],
        score_units=base['score_units'],
        score_scale=base['score_scale'],
        value_numeric=base['value_numeric'],
        value_eur=base['value_eur'],
        value_unit=base['value_unit'],
        team_keys=team_keys,
        team_raw_to_key=raw_to_key,
    )
    return prepared, mapped_matches, mapping_df

## 5. Reusable two-stage score optimizer

Data preparation is complete before this section runs. Each formation gets the same model-building function: maximize exact expected-point units, fix that optimum, then minimize integer-euro cost. Infeasible formations and individual solver failures are recorded without stopping the other formation evaluations.

In [5]:
# Handle cbc available for reuse in the workflow.
def ensure_cbc_available() -> None:
    solver = PULP_CBC_CMD(msg=False)
    # Validate the input before continuing with later processing.
    if not solver.available():
        raise RuntimeError(
            'PuLP is installed, but the CBC solver is unavailable. Install or configure CBC '
            'for this Python environment before running the optimizer.'
        )


# Handle formation for reuse in the workflow.
def solve_formation(
    prepared: PreparedData,
    mapped_matches: list[MappedMatch],
    formation: str,
    counts: dict[str, int],
    solver_msg: bool = False,
) -> FormationResult:
    model = LpProblem(f'Kickbase_Squad_{formation.replace("-", "_")}', LpMaximize)
    indices = [int(index) for index in prepared.df.index]
    x = {index: LpVariable(f'x_{index}', cat='Binary') for index in indices}
    captain = {index: LpVariable(f'captain_{index}', cat='Binary') for index in indices}

    squad_score_expression = lpSum(int(prepared.score_units.loc[index]) * x[index] for index in indices)
    captain_bonus_expression = lpSum(int(prepared.score_units.loc[index]) * captain[index] for index in indices)
    score_expression = squad_score_expression + captain_bonus_expression
    cost_expression = lpSum(int(prepared.value_eur.loc[index]) * x[index] for index in indices)
    model += score_expression, 'MaximizeScore'
    model += cost_expression <= BUDGET_EUR, 'Budget'
    model += lpSum(x[index] for index in indices) == SQUAD_SIZE, 'SquadCardinality'
    model += lpSum(captain[index] for index in indices) == 1, 'CaptainCardinality'
    # Process each available item while preserving the current workflow state.
    for index in indices:
        model += captain[index] <= x[index], f'CaptainMustBeSelected_{index}'
    model += lpSum(x[index] for index in indices if prepared.positions.loc[index] == 'GK') == 1, 'GKCount'
    # Process each available item while preserving the current workflow state.
    for position, required in counts.items():
        model += (
            lpSum(x[index] for index in indices if prepared.positions.loc[index] == position) == required,
            f'{position}Count',
        )

    # Process each available item while preserving the current workflow state.
    for team_key in sorted(prepared.team_keys.unique().tolist()):
        model += (
            lpSum(x[index] for index in indices if prepared.team_keys.loc[index] == team_key)
            <= MAX_PLAYERS_PER_CLUB,
            f'Club_{team_key}',
        )

    # Process each available item while preserving the current workflow state.
    for match in mapped_matches:
        model += (
            lpSum(
                x[index]
                for index in indices
                if prepared.team_keys.loc[index] in {match.home_key, match.away_key}
            ) <= MAX_PLAYERS_PER_MATCH,
            f'Match_{match.record.match_id}',
        )

    # Handle expected failures with a clear, actionable message.
    try:
        primary_code = model.solve(PULP_CBC_CMD(msg=solver_msg))
    except PulpSolverError as exc:
        return FormationResult(formation, f'Solver error (primary): {exc}')
    primary_status = LpStatus.get(primary_code, str(primary_code))
    if primary_status != 'Optimal':
        return FormationResult(formation, primary_status)

    primary_value = value(score_expression)
    if primary_value is None or not math.isfinite(float(primary_value)):
        return FormationResult(formation, 'Invalid primary objective value')
    best_score_units = int(round(float(primary_value)))
    model += score_expression == best_score_units, 'FixPrimaryOptimum'
    model.sense = LpMinimize
    model.setObjective(cost_expression)

    # Handle expected failures with a clear, actionable message.
    try:
        secondary_code = model.solve(PULP_CBC_CMD(msg=solver_msg))
    except PulpSolverError as exc:
        return FormationResult(formation, f'Solver error (cost tie-break): {exc}')
    secondary_status = LpStatus.get(secondary_code, str(secondary_code))
    if secondary_status != 'Optimal':
        return FormationResult(formation, f'Cost tie-break: {secondary_status}')

    chosen_indices = tuple(index for index in indices if (x[index].value() or 0.0) > 0.5)
    captain_indices = tuple(index for index in indices if (captain[index].value() or 0.0) > 0.5)
    if len(captain_indices) != 1 or captain_indices[0] not in chosen_indices:
        return FormationResult(formation, 'Invalid captain selection extracted from solver')
    captain_index = captain_indices[0]
    total_score_units = (
        sum(int(prepared.score_units.loc[index]) for index in chosen_indices)
        + int(prepared.score_units.loc[captain_index])
    )
    total_value_eur = sum(int(prepared.value_eur.loc[index]) for index in chosen_indices)
    if total_score_units != best_score_units:
        return FormationResult(formation, 'Extracted solution disagrees with primary optimum')
    return FormationResult(
        formation=formation,
        status='Optimal',
        chosen_indices=chosen_indices,
        captain_index=captain_index,
        total_score_units=total_score_units,
        total_value_eur=total_value_eur,
    )


# Handle all formations for reuse in the workflow.
def evaluate_all_formations(
    prepared: PreparedData, mapped_matches: list[MappedMatch], solver_msg: bool = False
) -> list[FormationResult]:
    ensure_cbc_available()
    results: list[FormationResult] = []
    # Process each available item while preserving the current workflow state.
    for formation, counts in ALLOWED_FORMATIONS.items():
        print(f'Solving formation {formation} ...')
        result = solve_formation(prepared, mapped_matches, formation, counts, solver_msg)
        print(f'  Status: {result.status}')
        results.append(result)
    return results


# Handle decimal for reuse in the workflow.
def score_decimal(units: int, scale: int) -> Decimal:
    return Decimal(units) / Decimal(scale)


# Format points for reuse in the workflow.
def format_score(units: int | None, scale: int) -> str:
    if units is None:
        return '—'
    return format(score_decimal(units, scale), 'f')


# Handle comparison for reuse in the workflow.
def formation_comparison(results: list[FormationResult], score_scale: int) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                'Formation': result.formation,
                'Solver Status': result.status,
                'Total Score (captain doubled)': format_score(result.total_score_units, score_scale),
                'Total Squad Value': (
                    f'€{result.total_value_eur:,}' if result.total_value_eur is not None else '—'
                ),
            }
            for result in results
        ]
    )


# Select global winner for reuse in the workflow.
def select_global_winner(results: list[FormationResult]) -> FormationResult:
    formation_order = {name: index for index, name in enumerate(ALLOWED_FORMATIONS)}
    feasible = [
        result
        for result in results
        if result.status == 'Optimal'
        and result.total_score_units is not None
        and result.total_value_eur is not None
    ]
    # Validate the input before continuing with later processing.
    if not feasible:
        statuses = '; '.join(f'{item.formation}: {item.status}' for item in results)
        raise RuntimeError(
            'No permitted formation produced a valid optimal solution; no CSV will be exported. '
            f'Statuses: {statuses}'
        )
    return min(
        feasible,
        key=lambda item: (
            -int(item.total_score_units),
            int(item.total_value_eur),
            formation_order[item.formation],
        ),
    )

## 6. Independent verification, readable ordering, and export

These functions verify the winning dataframe independently of PuLP's constraint objects, produce the requested readable order and console table, and export only original columns after all checks pass.

In [6]:
# Handle winning solution for reuse in the workflow.
def verify_winning_solution(
    winner: FormationResult, prepared: PreparedData, mapped_matches: list[MappedMatch]
) -> None:
    selected = list(winner.chosen_indices)
    failures: list[str] = []
    counts = ALLOWED_FORMATIONS[winner.formation]

    if len(selected) != SQUAD_SIZE:
        failures.append(f'expected {SQUAD_SIZE} players, found {len(selected)}')
    player_ids = prepared.df.loc[selected, prepared.columns['player_id']].astype(str).str.strip()
    if player_ids.nunique() != len(selected):
        failures.append('selected player IDs are not unique')

    selected_positions = prepared.positions.loc[selected]
    position_counts = selected_positions.value_counts().to_dict()
    if position_counts.get('GK', 0) != 1:
        failures.append(f'expected 1 GK, found {position_counts.get("GK", 0)}')
    # Process each available item while preserving the current workflow state.
    for position, required in counts.items():
        actual = position_counts.get(position, 0)
        if actual != required:
            failures.append(f'expected {required} {position}, found {actual}')

    total_value_eur = sum(int(prepared.value_eur.loc[index]) for index in selected)
    if total_value_eur > BUDGET_EUR:
        failures.append(f'squad value €{total_value_eur:,} exceeds €{BUDGET_EUR:,}')
    if winner.total_value_eur != total_value_eur:
        failures.append(
            f'recalculated value €{total_value_eur:,} disagrees with result €{winner.total_value_eur:,}'
        )

    club_counts = prepared.team_keys.loc[selected].value_counts().to_dict()
    over_club_limit = {
        team: count for team, count in club_counts.items() if count > MAX_PLAYERS_PER_CLUB
    }
    if over_club_limit:
        failures.append(f'club limit exceeded: {over_club_limit}')

    # Process each available item while preserving the current workflow state.
    for match in mapped_matches:
        match_count = sum(
            prepared.team_keys.loc[index] in {match.home_key, match.away_key}
            for index in selected
        )
        if match_count > MAX_PLAYERS_PER_MATCH:
            failures.append(f'match {match.record.match_id} contains {match_count} selected players')

    if winner.captain_index is None:
        failures.append('no captain was assigned')
    elif winner.captain_index not in selected:
        failures.append('captain is not in the selected squad')

    total_score_units = sum(int(prepared.score_units.loc[index]) for index in selected)
    if winner.captain_index is not None and winner.captain_index in selected:
        total_score_units += int(prepared.score_units.loc[winner.captain_index])
    if winner.total_score_units != total_score_units:
        failures.append(
            f'recalculated score units {total_score_units} disagree with result '
            f'{winner.total_score_units}'
        )
    numeric_total = float(prepared.score_numeric.loc[selected].sum())
    if winner.captain_index is not None and winner.captain_index in selected:
        numeric_total += float(prepared.score_numeric.loc[winner.captain_index])
    result_total = float(score_decimal(total_score_units, prepared.score_scale))
    if not math.isclose(numeric_total, result_total, rel_tol=1e-12, abs_tol=1e-9):
        failures.append(
            f'numeric score total {numeric_total} disagrees with exact total {result_total}'
        )

    # Validate the input before continuing with later processing.
    if failures:
        raise RuntimeError(
            'Post-solve verification failed; no CSV will be exported: ' + '; '.join(failures)
        )
    print('Independent post-solve verification passed.')


# Handle final squad for reuse in the workflow.
def sort_final_squad(
    winner: FormationResult, prepared: PreparedData
) -> tuple[pd.DataFrame, list[int]]:
    selected = list(winner.chosen_indices)
    position_order = {'GK': 0, 'DEF': 1, 'MID': 2, 'FOR': 3}
    helper = pd.DataFrame(index=selected)
    helper['position_order'] = prepared.positions.loc[selected].map(position_order).astype(int)
    helper['score'] = prepared.score_numeric.loc[selected]
    helper['name'] = (
        prepared.df.loc[selected, prepared.columns['player_name']].astype(str).str.casefold()
    )
    helper['player_id'] = prepared.df.loc[selected, prepared.columns['player_id']].astype(str)
    helper = helper.sort_values(
        ['position_order', 'score', 'name', 'player_id'],
        ascending=[True, False, True, True],
        kind='stable',
    )
    sorted_indices = [int(index) for index in helper.index]
    squad_df = prepared.df.loc[sorted_indices, prepared.original_columns].copy().reset_index(drop=True)
    return squad_df, sorted_indices


# Build selected player table for reuse in the workflow.
def build_selected_player_table(
    prepared: PreparedData, sorted_indices: list[int], captain_index: int
) -> pd.DataFrame:
    return pd.DataFrame(
        {
            'Full Name': prepared.df.loc[sorted_indices, prepared.columns['player_name']].tolist(),
            'Position': prepared.positions.loc[sorted_indices].tolist(),
            'Club': [TEAM_DISPLAY_NAMES[key] for key in prepared.team_keys.loc[sorted_indices]],
            'In-Game Value': prepared.df.loc[sorted_indices, prepared.columns['market_value']].tolist(),
            'Score': prepared.df.loc[sorted_indices, prepared.columns['score']].tolist(),
            'Captain': ['Yes' if index == captain_index else '' for index in sorted_indices],
        }
    )


# Handle verified squad for reuse in the workflow.
def export_verified_squad(
    squad_df: pd.DataFrame, metadata: ScoreMetadata
) -> Path:
    # Validate the input before continuing with later processing.
    if len(squad_df) != SQUAD_SIZE:
        raise RuntimeError(
            f'Refusing to export {len(squad_df)} rows; exactly {SQUAD_SIZE} are required.'
        )
    optimization_timestamp = datetime.now().astimezone().strftime(TIMESTAMP_FORMAT)
    output_name = (
        f'optimized_squad_{metadata.method}_{metadata.retrieval_timestamp}_'
        f'{metadata.metric_creation_timestamp}_{optimization_timestamp}.csv'
    )
    OPTIMIZED_SQUAD_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OPTIMIZED_SQUAD_DIR / output_name
    # Validate the input before continuing with later processing.
    if output_path.exists():
        raise FileExistsError(f'Refusing to overwrite existing optimized-squad CSV: {output_path}')
    # Handle expected failures with a clear, actionable message.
    try:
        squad_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    except OSError as exc:
        raise OSError(f'Could not save optimized squad to {output_path}: {exc}') from exc
    return output_path.resolve()

## 7. Run the complete workflow

Run this cell after the definition cells. It performs discovery, prompts for the matchday, prepares and maps data once, evaluates all formations, verifies the winner, prints the requested diagnostics, and writes the final CSV only after successful verification.

In [7]:
metadata = discover_latest_score(EXPECTED_POINTS_DIR)
print('Selected score input:')
print(f'  File: {metadata.path}')
print(f'  Retrieval timestamp: {metadata.retrieval_timestamp}')
print(f'  Method: {metadata.method}')
print(f'  Metric-creation timestamp: {metadata.metric_creation_timestamp}')

matchday = request_matchday()
matches, match_source, match_path = load_matchday_matches(matchday)
prepared, mapped_matches, mapping_table = prepare_optimization_data(metadata.path, matches)

print('\nMatch-to-club mapping diagnostics:')
print(mapping_table.to_string(index=False))
print(f'\nValidated {len(prepared.df):,} player rows once before formation solving.')
print(f'Detected market-value unit: {prepared.value_unit}; internal optimization unit: euros.')

formation_results = evaluate_all_formations(prepared, mapped_matches, solver_msg=False)
comparison_table = formation_comparison(formation_results, prepared.score_scale)
print('\nFormation comparison:')
display(comparison_table)

winner = select_global_winner(formation_results)
verify_winning_solution(winner, prepared, mapped_matches)
final_squad, sorted_indices = sort_final_squad(winner, prepared)
# Validate the input before continuing with later processing.
if list(final_squad.columns) != prepared.original_columns:
    raise RuntimeError('Final export columns do not exactly match the original CSV columns.')
selected_player_table = build_selected_player_table(prepared, sorted_indices, int(winner.captain_index))
output_path = export_verified_squad(final_squad, metadata)

total_score_text = format_score(winner.total_score_units, prepared.score_scale)
remaining_budget = BUDGET_EUR - int(winner.total_value_eur)

print('\n=== Input ===')
print(f'Score input file: {metadata.path}')
print(f'Retrieval timestamp: {metadata.retrieval_timestamp}')
print(f'Score method: {metadata.method}')
print(f'Metric-creation timestamp: {metadata.metric_creation_timestamp}')
print(f'Requested matchday: {matchday}')
print(f'Match JSON source used: {match_source}')
print(f'Match JSON path: {match_path}')
print(f'Detected/used market-value unit: {prepared.value_unit} -> integer euros')

print('\n=== Optimization result ===')
print(f'Chosen formation: {winner.formation}')
print(f'Solver status: {winner.status}')
print(f'Total score (captain doubled): {total_score_text}')
print(
    'Recommended captain: ' + str(
        prepared.df.loc[winner.captain_index, prepared.columns['player_name']]
    )
)
print(f'Total squad value: €{int(winner.total_value_eur):,}')
print(f'Remaining budget: €{remaining_budget:,}')

print('\n=== Selected players ===')
print(selected_player_table.to_string(index=False))

print('\n=== Output ===')
print(f'Optimized squad CSV: {output_path}')

Selected score input:
  File: C:\kickbase project\outputs\expected_points\expected_points_20260827_124042_+0200_sofascore_overall_rating_odds_lineup_20260827_124757_+0200.csv
  Retrieval timestamp: 20260827_124042_+0200
  Method: sofascore_overall_rating_odds_lineup
  Metric-creation timestamp: 20260827_124757_+0200


Enter the matchday to optimise the squad for:  1


Match source used: SofaScore (C:\kickbase project\outputs\sofascore\match_ids\match_ids_1.json)
Club mapping mode: embedded Kickbase team-ID bridge

Match-to-club mapping diagnostics:
 Match ID     JSON Home Team           CSV Home Club      JSON Away Team                 CSV Away Club
 16434087  FC Bayern München   FC Bayern München (2)       VfB Stuttgart             VfB Stuttgart (9)
 16434022         1. FC Köln         1. FC Köln (28)      TSG Hoffenheim           TSG Hoffenheim (14)
 16434026 1. FC Union Berlin 1. FC Union Berlin (40) Eintracht Frankfurt       Eintracht Frankfurt (4)
 16434044    1. FSV Mainz 05    1. FSV Mainz 05 (18)     SC Paderborn 07          SC Paderborn 07 (29)
 16434023         RB Leipzig         RB Leipzig (43) Borussia M'gladbach Borussia Mönchengladbach (15)
 16434020   SV 07 Elversberg   SV 07 Elversberg (77) Bayer 04 Leverkusen       Bayer 04 Leverkusen (7)
 16434025  Borussia Dortmund   Borussia Dortmund (3)        Hamburger SV              Hamburger

,Formation,Solver Status,Total Score (captain doubled),Total Squad Value
0,4-4-2,Optimal,164.322127,"€149,469,775"
1,4-2-4,Optimal,159.784339,"€147,092,336"
2,3-4-3,Optimal,162.953419,"€148,339,960"
3,4-3-3,Optimal,163.21241,"€149,871,175"
4,5-3-2,Optimal,163.994477,"€148,755,806"
5,3-5-2,Optimal,164.063136,"€147,938,560"
6,5-4-1,Optimal,164.628601,"€145,992,472"
7,4-5-1,Optimal,165.354379,"€149,925,126"
8,3-6-1,Optimal,164.766404,"€149,621,689"
9,5-2-3,Optimal,161.203579,"€149,013,901"


Independent post-solve verification passed.

=== Input ===
Score input file: C:\kickbase project\outputs\expected_points\expected_points_20260827_124042_+0200_sofascore_overall_rating_odds_lineup_20260827_124757_+0200.csv
Retrieval timestamp: 20260827_124042_+0200
Score method: sofascore_overall_rating_odds_lineup
Metric-creation timestamp: 20260827_124757_+0200
Requested matchday: 1
Match JSON source used: SofaScore
Match JSON path: C:\kickbase project\outputs\sofascore\match_ids\match_ids_1.json
Detected/used market-value unit: euros -> integer euros

=== Optimization result ===
Chosen formation: 4-5-1
Solver status: Optimal
Total score (captain doubled): 165.354379
Recommended captain: Dominik Kohr
Total squad value: €149,925,126
Remaining budget: €74,874

=== Selected players ===
            Full Name Position                Club In-Game Value     Score Captain
         Manuel Neuer       GK   FC Bayern München    13764324.0 16.719669        
         Dominik Kohr      DEF     1. F

## 8. Select this lineup

Select a verified result only when it should become this league's canonical lineup.


In [8]:
# Selected-lineup persistence is shared by every squad workflow.
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from selected_lineups import make_selected_lineup, select_lineup_interactively


def build_selected_lineup_snapshot(
    league: str,
    source: str,
    prepared: PreparedData,
    sorted_indices: list[int],
    captain_index: int,
    total_score_units: int,
    metadata: ScoreMetadata,
    matchday: int,
    formation: str,
) -> dict[str, Any]:
    """Create a portable selected-lineup snapshot from verified notebook data."""
    players = []
    for index in sorted_indices:
        team_key = prepared.team_keys.loc[index]
        players.append(
            {
                'id': prepared.df.loc[index, prepared.columns['player_id']],
                'name': prepared.df.loc[index, prepared.columns['player_name']],
                'position': prepared.positions.loc[index],
                'market_value': prepared.df.loc[index, prepared.columns['market_value']],
                'market_value_eur': int(prepared.value_eur.loc[index]),
                'club': TEAM_DISPLAY_NAMES[team_key],
                'expected_points': prepared.df.loc[index, prepared.columns['score']],
                'captain': index == captain_index,
            }
        )
    return make_selected_lineup(
        league=league,
        players=players,
        expected_points={
            'value': format_score(total_score_units, prepared.score_scale),
            'label': 'Total expected points (captain doubled)',
            'includes_captain_bonus': True,
        },
        source=source,
        player_count=SQUAD_SIZE,
        metadata={
            'score_input_file': str(metadata.path),
            'score_method': metadata.method,
            'metric_creation_timestamp': metadata.metric_creation_timestamp,
            'matchday': matchday,
            'formation': formation,
        },
    )


In [9]:


# Selection is intentionally separate from the timestamped optimizer export.
selected_lineup_snapshot = build_selected_lineup_snapshot(
    league='All Limits Arena',
    source='optimizer_all_limits_arena',
    prepared=prepared,
    sorted_indices=sorted_indices,
    captain_index=int(winner.captain_index),
    total_score_units=int(winner.total_score_units),
    metadata=metadata,
    matchday=matchday,
    formation=winner.formation,
)
selected_lineup_path = select_lineup_interactively(
    selected_lineup_snapshot, filename='all-limits-arena.json'
)


Select this lineup for All Limits Arena? [y/n]:  n


Lineup was not selected; the existing selection is unchanged.


In [10]:
from project_paths import prune_timestamped_outputs

removed_outputs = prune_timestamped_outputs()
print(f"Pruned {len(removed_outputs)} expired timestamped output(s).")


Pruned 1 expired timestamped output(s).
